In [1]:
import sys, os
import subprocess
import pandas as pd

from common_utils.utils import chunks
from tqdm import tqdm
from matplotlib import pyplot as plt
from pylab import rcParams
import seaborn as sns

from IPython.display import display, HTML
import pandas as pd
import numpy as np

import logging
from datetime import datetime, timedelta
import pandas as pd

import requests
import pytz

import io

In [14]:
from dbutils.query_utils import run_select

from dbutils.query_utils import get_interval_clauses, run_select
from dbutils.utils import get_nabu_payments_client, get_nabu_ledger_client
from dbutils.query_utils import get_select_query, run_select, get_interval_clauses
from customers_analytics.fraud_analysis.fraud_db_utils import get_fraud_client
from customers_analytics.nabu.nabu_db_utils import (
    get_nabu_client_internal, 
    get_nabu_backoffice_client, 
    get_nabu_client_private, 
    get_nabu_backoffice_client_prod, 
    categorize_blockchain_error_reasons
)

INFO:root:Role:data-science Environment:internal


In [5]:
dropbox_lnk_xlsx = 'https://www.dropbox.com/scl/fi/dt0spyl7ny3s0m72o9sh4/Fraud-Data.xlsx?dl=1&rlkey=l5s82jh8cvotpvbd16fs37gjs'
dropbox_lnk_csv  ='https://www.dropbox.com/s/bptgqsfnquv89qt/Fraud%20Data.csv?dl=1'

## Check different approaches to read data from external links

In [7]:
pd.read_excel(dropbox_lnk_xlsx, sheet_name='New Fraud Data').head()

,Ticket week,Ticket tags,Metrics,Year,Month,TRXN Week,TRXN date,TRXN volume,TRXN value (€),TRXN LHV Ref,Account Servicer Reference
0,15,Blockchain,SEPA,2022,April,14,2022-07-04 00:00:00,1,1000.0,502420401,400B4B9E-7FB6-EC11-A2E4-00155DA95D0B
1,15,Blockchain,SEPA,2022,April,14,2022-07-04 00:00:00,1,10000.0,502420397,3D0B4B9E-7FB6-EC11-A2E4-00155DA95D0B
2,15,Blockchain,SEPA,2022,April,14,2022-07-04 00:00:00,1,500.0,502420430,5C0B4B9E-7FB6-EC11-A2E4-00155DA95D0B
3,15,Blockchain,SEPA,2022,April,14,2022-07-04 00:00:00,1,500.0,502420432,5E0B4B9E-7FB6-EC11-A2E4-00155DA95D0B
4,15,Blockchain,SEPA,2022,April,14,2022-07-04 00:00:00,1,3000.0,502420395,3A0B4B9E-7FB6-EC11-A2E4-00155DA95D0B


In [19]:
s=requests.get(dropbox_lnk_xlsx).content
with io.BytesIO(s) as fh:
    df = pd.read_excel(fh, sheet_name=0, engine='openpyxl')

df.head()

,Ticket week,Ticket tags,Metrics,Year,Month,TRXN Week,TRXN date,TRXN volume,TRXN value (€),TRXN LHV Ref,Account Servicer Reference
0,15,Blockchain,SEPA,2022,April,14,2022-07-04 00:00:00,1,1000.0,502420401,400B4B9E-7FB6-EC11-A2E4-00155DA95D0B
1,15,Blockchain,SEPA,2022,April,14,2022-07-04 00:00:00,1,10000.0,502420397,3D0B4B9E-7FB6-EC11-A2E4-00155DA95D0B
2,15,Blockchain,SEPA,2022,April,14,2022-07-04 00:00:00,1,500.0,502420430,5C0B4B9E-7FB6-EC11-A2E4-00155DA95D0B
3,15,Blockchain,SEPA,2022,April,14,2022-07-04 00:00:00,1,500.0,502420432,5E0B4B9E-7FB6-EC11-A2E4-00155DA95D0B
4,15,Blockchain,SEPA,2022,April,14,2022-07-04 00:00:00,1,3000.0,502420395,3A0B4B9E-7FB6-EC11-A2E4-00155DA95D0B


In [266]:
s=requests.get(dropbox_lnk_csv).content
pd.read_csv(io.StringIO(s.decode('utf-8', errors='ignore'))).head()

,Ticket week,Ticket tags,Metrics,Year,Month,TRXN Week,TRXN date,TRXN volume,TRXN value (€),TRXN LHV Ref,Account Servicer Reference
0,15,Blockchain,SEPA,2022,April,14,2022-07-04 00:00:00,1,1000.0,502420401,400B4B9E-7FB6-EC11-A2E4-00155DA95D0B
1,15,Blockchain,SEPA,2022,April,14,2022-07-04 00:00:00,1,10000.0,502420397,3D0B4B9E-7FB6-EC11-A2E4-00155DA95D0B
2,15,Blockchain,SEPA,2022,April,14,2022-07-04 00:00:00,1,500.0,502420430,5C0B4B9E-7FB6-EC11-A2E4-00155DA95D0B
3,15,Blockchain,SEPA,2022,April,14,2022-07-04 00:00:00,1,500.0,502420432,5E0B4B9E-7FB6-EC11-A2E4-00155DA95D0B
4,15,Blockchain,SEPA,2022,April,14,2022-07-04 00:00:00,1,3000.0,502420395,3A0B4B9E-7FB6-EC11-A2E4-00155DA95D0B


In [233]:
cmd = "wget --no-check-cert -O ~/data-ops/csv/fraud_lhv.xlsx '{}'".format(dropbox_lnk)

ps = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
output = ps.communicate()[0]
print(output)


b'--2022-05-04 10:12:21--  https://www.dropbox.com/scl/fi/dt0spyl7ny3s0m72o9sh4/Fraud-Data.xlsx?dl=1&rlkey=l5s82jh8cvotpvbd16fs37gjs\nResolving www.dropbox.com (www.dropbox.com)... 162.125.66.18, 2620:100:6023:18::a27d:4312\nConnecting to www.dropbox.com (www.dropbox.com)|162.125.66.18|:443... connected.\nHTTP request sent, awaiting response... 302 Found\nLocation: https://uc3652c90970e10084bbae5181b7.dl.dropboxusercontent.com/cd/0/get/BkkLAJ_iSTE1E1ON5POGeeMtTQRpikX0PPL7UYM15K_uEHhbCrSkvqIieoXN-K7RFVsh2GtcUvZ_cwbzR4WJrLndWWqSCYuuhimzXyWqBZpEM-c43v6bASYMZlEq6PUrXw7QuP5gsFOTe3yIe56kQdgkNl7lhaALpP9zrs780mC5611oiJQ5lDSrstWjQoqBHj4/file?dl=1# [following]\n--2022-05-04 10:12:21--  https://uc3652c90970e10084bbae5181b7.dl.dropboxusercontent.com/cd/0/get/BkkLAJ_iSTE1E1ON5POGeeMtTQRpikX0PPL7UYM15K_uEHhbCrSkvqIieoXN-K7RFVsh2GtcUvZ_cwbzR4WJrLndWWqSCYuuhimzXyWqBZpEM-c43v6bASYMZlEq6PUrXw7QuP5gsFOTe3yIe56kQdgkNl7lhaALpP9zrs780mC5611oiJQ5lDSrstWjQoqBHj4/file?dl=1\nResolving uc3652c90970e10084bbae5181

In [231]:
cmd = "curl -L -I '{}' -o Fraud-Data.xlsx".format(dropbox_lnk)

ps = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
output = ps.communicate()[0]
print(output)

b'  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current\n                                 Dload  Upload   Total   Spent    Left  Speed\n\r  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0\r  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0\r  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0\n\r  0 39616    0     0    0     0      0      0 --:--:--  0:00:01 --:--:--     0\n'


In [234]:
pd.read_excel('~/data-ops/csv/fraud_lhv.xlsx')

,Ticket week,Ticket tags,Metrics,Year,Month,TRXN Week,TRXN date,TRXN volume,TRXN value (€),TRXN LHV Ref,Account Servicer Reference
0,15,Blockchain,SEPA,2022,April,14,2022-07-04 00:00:00,1,1000.00,502420401,400B4B9E-7FB6-EC11-A2E4-00155DA95D0B
1,15,Blockchain,SEPA,2022,April,14,2022-07-04 00:00:00,1,10000.00,502420397,3D0B4B9E-7FB6-EC11-A2E4-00155DA95D0B
2,15,Blockchain,SEPA,2022,April,14,2022-07-04 00:00:00,1,500.00,502420430,5C0B4B9E-7FB6-EC11-A2E4-00155DA95D0B
3,15,Blockchain,SEPA,2022,April,14,2022-07-04 00:00:00,1,500.00,502420432,5E0B4B9E-7FB6-EC11-A2E4-00155DA95D0B
4,15,Blockchain,SEPA,2022,April,14,2022-07-04 00:00:00,1,3000.00,502420395,3A0B4B9E-7FB6-EC11-A2E4-00155DA95D0B
...,...,...,...,...,...,...,...,...,...,...,...
105,16,Blockchain,SEPA,2022,April,16,19/04/2022,1,5000.00,507846925,62EC1857-A4BF-EC11-A2E4-00155DA95D0B
106,16,Blockchain,SEPA,2022,April,16,19/04/2022,1,5000.00,507847018,BFEC1857-A4BF-EC11-A2E4-00155DA95D0B
107,16,Blockchain,SEPA,2022,April,16,19/04/2022,1,4963.63,507847045,DBEC1857-A4BF-EC11-A2E4-00155DA95D0B
108,16,Blockchain,SEPA,2022,April,16,19/04/2022,1,4985.52,507847128,2CED1857-A4BF-EC11-A2E4-00155DA95D0B


## Prepare external table with fraud references

In [155]:
import re

In [150]:
def convert_to_datetime(x, milliseconds=True):
    if not hasattr(x, 'minute'):
        x = pd.to_datetime(x, utc = True)
    else:
        x = x.astimezone(tz=pytz.UTC)
        
    epoch = pytz.UTC.localize(datetime.utcfromtimestamp(0))
    delta = x - epoch

    result = delta.total_seconds()
    if milliseconds:
        result *= 1000

    res = pytz.UTC.localize(datetime.utcfromtimestamp(result/1e3)) 

    return res

In [156]:
df = pd.read_excel("csv/Fraud Data.xlsx")

In [148]:
df.head()

,Ticket week,Ticket tags,Metrics,Year,Month,TRXN Week,TRXN date,TRXN volume,TRXN value (€),TRXN LHV Ref,Account Servicer Reference
0,15,Blockchain,SEPA,2022,April,14,2022-07-04 00:00:00,1,1000.0,502420401,400B4B9E-7FB6-EC11-A2E4-00155DA95D0B
1,15,Blockchain,SEPA,2022,April,14,2022-07-04 00:00:00,1,10000.0,502420397,3D0B4B9E-7FB6-EC11-A2E4-00155DA95D0B
2,15,Blockchain,SEPA,2022,April,14,2022-07-04 00:00:00,1,500.0,502420430,5C0B4B9E-7FB6-EC11-A2E4-00155DA95D0B
3,15,Blockchain,SEPA,2022,April,14,2022-07-04 00:00:00,1,500.0,502420432,5E0B4B9E-7FB6-EC11-A2E4-00155DA95D0B
4,15,Blockchain,SEPA,2022,April,14,2022-07-04 00:00:00,1,3000.0,502420395,3A0B4B9E-7FB6-EC11-A2E4-00155DA95D0B


## Work with date column: cast all variations to datetime with defined UTC tz

In [149]:
df['TRXN date'].unique().tolist()

[datetime.datetime(2022, 7, 4, 0, 0),
 datetime.datetime(2022, 4, 4, 0, 0),
 datetime.datetime(2022, 8, 4, 0, 0),
 datetime.datetime(2022, 5, 4, 0, 0),
 '21/03/2022',
 datetime.datetime(2022, 11, 4, 0, 0),
 datetime.datetime(2022, 1, 4, 0, 0),
 datetime.datetime(2022, 12, 4, 0, 0),
 '22/03/2022',
 '24/03/2022',
 datetime.datetime(2022, 3, 4, 0, 0),
 '13/04/2022',
 '14/04/2022',
 datetime.datetime(2022, 8, 3, 0, 0),
 '16/04/2022',
 '19/04/2022',
 '20/04/2022',
 '21/04/2022',
 datetime.datetime(2022, 10, 4, 0, 0),
 '18/04/2022']

In [151]:
df['TRXN date'] = df['TRXN date'].apply(lambda x: convert_to_datetime(x))

In [152]:
df['TRXN date'].min(), df['TRXN date'].max()

(Timestamp('2022-01-04 00:00:00+0000', tz='UTC'),
 Timestamp('2022-12-04 00:00:00+0000', tz='UTC'))

In [154]:
df['TRXN date'].unique()

<DatetimeArray>
['2022-07-04 00:00:00+00:00', '2022-04-04 00:00:00+00:00',
 '2022-08-04 00:00:00+00:00', '2022-05-04 00:00:00+00:00',
 '2022-03-21 00:00:00+00:00', '2022-11-04 00:00:00+00:00',
 '2022-01-04 00:00:00+00:00', '2022-12-04 00:00:00+00:00',
 '2022-03-22 00:00:00+00:00', '2022-03-24 00:00:00+00:00',
 '2022-03-04 00:00:00+00:00', '2022-04-13 00:00:00+00:00',
 '2022-04-14 00:00:00+00:00', '2022-08-03 00:00:00+00:00',
 '2022-04-16 00:00:00+00:00', '2022-04-19 00:00:00+00:00',
 '2022-04-20 00:00:00+00:00', '2022-04-21 00:00:00+00:00',
 '2022-10-04 00:00:00+00:00', '2022-04-18 00:00:00+00:00']
Length: 20, dtype: datetime64[ns, UTC]

## Export to csv

In [60]:
df.to_csv("csv/Fraud Data.csv", index= False, encoding = "utf-8")

## Rename Non-ASCII symbols

In [41]:
del_chars = " ".join([chr(i) for i in list(range(32)) + list(range(127, 256))])
trans = str.maketrans(del_chars, " "*len(del_chars))

col_renames = {}

for col in df.columns:
    col_renames[col] = col.translate(trans).encode('utf-8', 'ignore').decode('utf-8')   
    col_renames[col] = re.sub(r'[^\w\s;,.\:\-#0-9]', '', col, flags=re.UNICODE)

In [42]:
col_renames

{'Ticket week': 'Ticket week',
 'Ticket tags': 'Ticket tags',
 'Metrics': 'Metrics',
 'Year': 'Year',
 'Month': 'Month',
 'TRXN Week': 'TRXN Week',
 'TRXN date': 'TRXN date',
 'TRXN volume': 'TRXN volume',
 'TRXN value (€)': 'TRXN value ',
 'TRXN LHV Ref': 'TRXN LHV Ref',
 'Account Servicer Reference': 'Account Servicer Reference'}

In [43]:
df.rename(columns=col_renames, inplace=True)

In [44]:
df.head()

,Ticket week,Ticket tags,Metrics,Year,Month,TRXN Week,TRXN date,TRXN volume,TRXN value,TRXN LHV Ref,Account Servicer Reference
0,15,Blockchain,SEPA,2022,April,14,2022-07-04 00:00:00,1,1000.0,502420401,400B4B9E-7FB6-EC11-A2E4-00155DA95D0B
1,15,Blockchain,SEPA,2022,April,14,2022-07-04 00:00:00,1,10000.0,502420397,3D0B4B9E-7FB6-EC11-A2E4-00155DA95D0B
2,15,Blockchain,SEPA,2022,April,14,2022-07-04 00:00:00,1,500.0,502420430,5C0B4B9E-7FB6-EC11-A2E4-00155DA95D0B
3,15,Blockchain,SEPA,2022,April,14,2022-07-04 00:00:00,1,500.0,502420432,5E0B4B9E-7FB6-EC11-A2E4-00155DA95D0B
4,15,Blockchain,SEPA,2022,April,14,2022-07-04 00:00:00,1,3000.0,502420395,3A0B4B9E-7FB6-EC11-A2E4-00155DA95D0B


In [45]:
df.to_csv("csv/Fraud Data.csv", index= False, encoding = "utf-8")

## Check Data from csv format

In [61]:
pd.read_csv("csv/Fraud Data.csv", encoding = "utf-8", engine='python').head()

,Ticket week,Ticket tags,Metrics,Year,Month,TRXN Week,TRXN date,TRXN volume,TRXN value (€),TRXN LHV Ref,Account Servicer Reference
0,15,Blockchain,SEPA,2022,April,14,2022-07-04 00:00:00,1,1000.0,502420401,400B4B9E-7FB6-EC11-A2E4-00155DA95D0B
1,15,Blockchain,SEPA,2022,April,14,2022-07-04 00:00:00,1,10000.0,502420397,3D0B4B9E-7FB6-EC11-A2E4-00155DA95D0B
2,15,Blockchain,SEPA,2022,April,14,2022-07-04 00:00:00,1,500.0,502420430,5C0B4B9E-7FB6-EC11-A2E4-00155DA95D0B
3,15,Blockchain,SEPA,2022,April,14,2022-07-04 00:00:00,1,500.0,502420432,5E0B4B9E-7FB6-EC11-A2E4-00155DA95D0B
4,15,Blockchain,SEPA,2022,April,14,2022-07-04 00:00:00,1,3000.0,502420395,3A0B4B9E-7FB6-EC11-A2E4-00155DA95D0B


## LHV Ingesting Manual Check

In [9]:
# from customers_analytics.nabu.internal_tables.bank_payments_fraud_stats import ingest_lhv_data

In [62]:
COLUMNS = [
    "id",
    "user_id",
    "created_at",
    "updated_at",
    "usd_amount",
    "external_ref",
    "modulr_customer_id",
    "is_fraud",
    "partner",
]

In [63]:
def _ingest_lhv_data(path):
    """
    Parameters
    ----------
    path: path to CSV with reported fraud cases. Needs to have and LHV external reference column, called external_ref
    """
    df = pd.read_csv(path, encoding = "utf-8", engine='python')
    df.rename(columns={"Account Servicer Reference": "external_ref"}, inplace=True)
    
    if "external_ref" not in df.columns:
        raise RuntimeError("external_ref not in dataframe's columns. Fix the data and retry.")
    df = df.loc[df["external_ref"].notnull()]
    df["external_ref"] = df["external_ref"].apply(lambda x: x.replace("-", "").strip(" "))
    df = df[df["external_ref"].notnull()].drop_duplicates("external_ref")

    # Get LHV transactions with those external_refs
    client = get_nabu_payments_client()
    transactions = run_select(
        client=client,
        table="payments.transaction_view",
        cols=[
            "payments.transaction_view.id as id",
            "payments.transaction_view.user as user_id",
            "payments.transaction_view.created_at as created_at",
            "payments.transaction_view.state_created_at as updated_at",
            "payments.transaction_view.external_ref as external_ref",
            "payments.transaction_view.usd_amount as usd_amount",
            "payments.transaction_view.partner",
        ],
        in_clauses={
            "payments.transaction_view.external_ref": (df["external_ref"], True),
        },
    )

    transactions["modulr_customer_id"] = None
    transactions["is_fraud"] = True

    assert set(transactions.columns) == set(
        COLUMNS
    ), "Columns in dataframe are different than columns in database. Check."

    transactions = transactions.sort_values("created_at").drop_duplicates("id", keep="first")
    transactions["updated_at"] = datetime.utcnow()
    
    return transactions

In [64]:
lhv_fraud_trxn = _ingest_lhv_data('csv/Fraud Data.csv')

In [208]:
lhv_fraud_trxn.external_ref.values

array(['CBA570DEAB9EEC11A2E400155DA95D0B',
       '2DCC31BAFCA8EC11A2E400155DA95D0B',
       'C990BCD8FCA8EC11A2E400155DA95D0B',
       '4F0E93E71EA9EC11A2E400155DA95D0B',
       '520E93E71EA9EC11A2E400155DA95D0B',
       '9EF855F025A9EC11A2E400155DA95D0B',
       'E334F2D02EA9EC11A2E400155DA95D0B',
       '8FCDAC35E9A9EC11A2E400155DA95D0B',
       'C8BC75B743ABEC11A2E400155DA95D0B',
       'CBDADB6347ABEC11A2E400155DA95D0B',
       '2C452562A0B1EC11A2E400155DA95D0B',
       '92FEEF9DB5B1EC11A2E400155DA95D0B',
       '78D0A62AB7B1EC11A2E400155DA95D0B',
       'E6351DFEC6B1EC11A2E400155DA95D0B',
       '8EF3491F0DB3EC11A2E400155DA95D0B',
       '0EED25D524B4EC11A2E400155DA95D0B',
       '15ED25D524B4EC11A2E400155DA95D0B',
       '607D9502EEB4EC11A2E400155DA95D0B',
       '3D0B4B9E7FB6EC11A2E400155DA95D0B',
       '3A0B4B9E7FB6EC11A2E400155DA95D0B',
       '400B4B9E7FB6EC11A2E400155DA95D0B',
       '5A0B4B9E7FB6EC11A2E400155DA95D0B',
       '5C0B4B9E7FB6EC11A2E400155DA95D0B',
       '5E0

In [65]:
lhv_fraud_trxn.shape

(110, 9)

In [157]:
lhv_fraud_trxn.head()

,id,user_id,created_at,updated_at,external_ref,usd_amount,partner,modulr_customer_id,is_fraud
89,d58ab484-053d-4feb-aa6b-471370bff8dd,11ab7eb6-c16f-46b1-8241-ec9ba57e8b24,2022-03-08 06:56:43.155,2022-05-03 16:19:20.214662,CBA570DEAB9EEC11A2E400155DA95D0B,2172.000000000,LHV,None,True
19,987930c7-31ba-4459-923b-81b40b86a212,fa6b5772-76d1-4c31-bcb5-59ced2603219,2022-03-21 09:53:32.384,2022-05-03 16:19:20.214662,2DCC31BAFCA8EC11A2E400155DA95D0B,2688.686000000,LHV,None,True
85,2120ad5e-b064-47a8-83d9-a01d98d7b822,fa6b5772-76d1-4c31-bcb5-59ced2603219,2022-03-21 09:54:24.827,2022-05-03 16:19:20.214662,C990BCD8FCA8EC11A2E400155DA95D0B,2761.814720000,LHV,None,True
32,3071945d-5a64-4287-ba78-7a19f3234264,fa6b5772-76d1-4c31-bcb5-59ced2603219,2022-03-21 13:58:33.605,2022-05-03 16:19:20.214662,4F0E93E71EA9EC11A2E400155DA95D0B,1656.000000000,LHV,None,True
35,1e9190c9-6119-407c-aae4-a9762aaf0dcd,fa6b5772-76d1-4c31-bcb5-59ced2603219,2022-03-21 13:58:34.405,2022-05-03 16:19:20.214662,520E93E71EA9EC11A2E400155DA95D0B,1021.200000000,LHV,None,True


In [164]:
lhv_fraud_trxn.to_excel("csv/lhv_fraud_trxn.xlsx", index=False)

In [207]:
lhv_fraud_trxn.to_csv("csv/lhv_fraud_trxn.csv", index=False)

## Work on bank_payments_fraud_stats

In [158]:
client_internal = get_nabu_client_internal()

In [160]:
query = \
'''
select * from bank_payments_fraud_stats
where updated_at >= '2022-04-01T00:00:00+00:00'
'''

fraud_stats = client_internal.dataframe_from_query(query)

In [162]:
fraud_stats.shape

(59301, 9)

In [161]:
fraud_stats.head()

,id,user_id,created_at,updated_at,external_ref,usd_amount,modulr_customer_id,is_fraud,partner
0,2f9c772e-209c-45fd-8e32-8c99e9961c37,7d60a18e-5514-4652-8819-8c2777b7dea8,2020-12-09 22:25:09,2022-01-20 21:51:19.795527,T2104BNGVY,1345.440000,C213FE5C,True,MODULR
1,c44bf480-3791-4f4e-950a-a561d01a669f,dcff2709-ffb1-4ba3-a282-80bef1a19856,2020-12-21 14:09:09,2022-01-20 21:51:19.795527,T2104FUTG4,202.022187,C213HHSR,True,MODULR
2,5fb69abe-7ed7-40e3-95a2-f7c3b5125ad6,079f8ee9-779e-45fe-a9a4-4c19c5bbbd4d,2020-12-23 09:17:06,2022-01-20 21:51:19.795527,T2104GHG81,1611.313468,C213H44E,True,MODULR
3,34bcf539-9819-4b7d-ab06-f77491a2238c,079f8ee9-779e-45fe-a9a4-4c19c5bbbd4d,2020-12-25 19:28:05,2022-01-20 21:51:19.795527,T2104HFNQD,6110.418175,C213H44E,True,MODULR
4,19889b3c-5fc4-490f-b059-181ed3c8a280,079f8ee9-779e-45fe-a9a4-4c19c5bbbd4d,2020-12-26 10:16:32,2022-01-20 21:51:19.795527,T2104HHWJ3,4073.612117,C213H44E,True,MODULR


## Upsert LHV_TRNX with `bank_payments_fraud_stats`

In [168]:
lhv_updated_fraud_stats = fraud_stats.merge(lhv_fraud_trxn, on=['id'], how='outer', indicator=True, validate="1:1")

In [169]:
lhv_updated_fraud_stats.shape

(59301, 18)

In [216]:
lhv_updated_fraud_stats.apply()

0        1345.440000
1         202.022187
2        1611.313468
3        6110.418175
4        4073.612117
            ...     
59296     187.500000
59297     686.250000
59298    1722.000000
59299      62.500000
59300      59.850000
Name: usd_amount_x, Length: 59301, dtype: float64

In [172]:
lhv_updated_fraud_stats[lhv_updated_fraud_stats['_merge'] == 'both'].shape

(110, 18)

## Filter coincidented rows

In [187]:
merge_both = lhv_updated_fraud_stats[lhv_updated_fraud_stats['_merge'] == 'both']

In [188]:
merge_both = merge_both.reset_index(drop=True)

In [194]:
merge_both.index.values

array([  0,   1,   2,   3,   4,   5,   6,   7,   8,   9,  10,  11,  12,
        13,  14,  15,  16,  17,  18,  19,  20,  21,  22,  23,  24,  25,
        26,  27,  28,  29,  30,  31,  32,  33,  34,  35,  36,  37,  38,
        39,  40,  41,  42,  43,  44,  45,  46,  47,  48,  49,  50,  51,
        52,  53,  54,  55,  56,  57,  58,  59,  60,  61,  62,  63,  64,
        65,  66,  67,  68,  69,  70,  71,  72,  73,  74,  75,  76,  77,
        78,  79,  80,  81,  82,  83,  84,  85,  86,  87,  88,  89,  90,
        91,  92,  93,  94,  95,  96,  97,  98,  99, 100, 101, 102, 103,
       104, 105, 106, 107, 108, 109])

In [189]:
np.where(merge_both['is_fraud_x']==merge_both['is_fraud_y'])[0].shape

(105,)

In [199]:
diff = merge_both.loc[np.where(merge_both['is_fraud_x']!=merge_both['is_fraud_y'])[0],:]

In [200]:
diff.head()

,id,user_id_x,created_at_x,updated_at_x,external_ref_x,usd_amount_x,modulr_customer_id_x,is_fraud_x,partner_x,user_id_y,created_at_y,updated_at_y,external_ref_y,usd_amount_y,partner_y,modulr_customer_id_y,is_fraud_y,_merge
26,f02b06ea-144a-4020-af10-adf97c8a28a3,21201a8c-5927-4415-8b42-068910103cc9,2022-04-12 05:50:58.419,2022-04-12 05:50:58.899436,5B836C8723BAEC11A2E400155DA95D0B,5450.0000,None,False,LHV,21201a8c-5927-4415-8b42-068910103cc9,2022-04-12 05:50:58.419,2022-05-03 16:19:20.214662,5B836C8723BAEC11A2E400155DA95D0B,5435.000000000,LHV,None,True,both
52,2c016fad-a87d-4904-aa45-973989c419bf,6450ad2d-4c29-482f-b0ae-46da4e9bb068,2022-04-20 05:52:38.109,2022-04-20 05:52:38.587005,459821DE6CC0EC11A2E400155DA95D0B,1.0692,None,False,LHV,6450ad2d-4c29-482f-b0ae-46da4e9bb068,2022-04-20 05:52:38.109,2022-05-03 16:19:20.214662,459821DE6CC0EC11A2E400155DA95D0B,1.071180000,LHV,None,True,both
53,cb43d8fa-6d00-4129-ba4c-7aa6777105a3,6450ad2d-4c29-482f-b0ae-46da4e9bb068,2022-04-20 05:52:49.164,2022-04-20 05:52:49.630975,D49821DE6CC0EC11A2E400155DA95D0B,1.0692,None,False,LHV,6450ad2d-4c29-482f-b0ae-46da4e9bb068,2022-04-20 05:52:49.164,2022-05-03 16:19:20.214662,D49821DE6CC0EC11A2E400155DA95D0B,1.071180000,LHV,None,True,both
54,43082b65-627d-4372-bfc3-28554be48f69,6450ad2d-4c29-482f-b0ae-46da4e9bb068,2022-04-20 05:53:00.275,2022-04-20 05:53:00.758711,309921DE6CC0EC11A2E400155DA95D0B,10.7460,None,False,LHV,6450ad2d-4c29-482f-b0ae-46da4e9bb068,2022-04-20 05:53:00.275,2022-05-03 16:19:20.214662,309921DE6CC0EC11A2E400155DA95D0B,10.765900000,LHV,None,True,both
62,23b818cc-7d15-4679-ad75-01e293434861,dd25f6d3-a6e2-4805-8ab3-e6750eb87cc6,2022-04-21 05:45:31.046,2022-04-21 05:45:31.691640,B599D9EC35C1EC11A2E400155DA95D0B,5400.0000,None,False,LHV,dd25f6d3-a6e2-4805-8ab3-e6750eb87cc6,2022-04-21 05:45:31.046,2022-05-03 16:19:20.214662,B599D9EC35C1EC11A2E400155DA95D0B,5425.000000000,LHV,None,True,both


In [206]:
diff.to_excel('csv/merge_diff_lhv_vs_nabu_fraud_stats.xlsx', index=False, )

In [205]:
diff.to_csv('csv/merge_diff_lhv_vs_nabu_fraud_stats.csv', index=False, )